# Fruits Vector Demo — Stage 2: Similarity Metrics

With the LanceDB table from Stage 1 ready, we now load the stored fruit vectors, reissue the mango query, and compare cosine, Euclidean, and dot-product rankings to show how the neighbour order shifts while the winner remains stable.

**What you'll learn:**
- How different similarity metrics work
- Visualize fruit vectors in 3D space
- See why some metrics give different rankings

## 1) Reload the stored fruit vectors

This cell reopens the LanceDB path, reads every fruit record, and prepares the mango query so the notebook works independently without rerunning Stage 1 cells.

In [ ]:
import numpy as np
import lancedb
from pathlib import Path

DB_PATH = Path("fruits_lancedb")
TABLE_NAME = "fruit_vectors"

db = lancedb.connect(str(DB_PATH))
table = db.open_table(TABLE_NAME)
stored = table.to_pandas()
fruit_vectors = {row.name: np.array(row.vector) for row in stored.itertuples()}
mango_query = np.array([0.9, 0.34, 0.08])

print(f"✅ Reloaded {len(fruit_vectors)} fruits from LanceDB")
print(f"✅ Mango query vector: {mango_query}")
print("\n📊 Stored fruits:")
for name, vec in fruit_vectors.items():
    print(f"  {name:12} → {vec}")

## 2) Visualize Fruit Vectors in 3D Space

Let's plot the fruits in 3D space using their color vectors. The mango query vector is shown in red.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Prepare data for plotting
names = list(fruit_vectors.keys())
vectors = np.array(list(fruit_vectors.values()))

fig = plt.figure(figsize=(12, 5))

# Plot 1: 3D scatter plot
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(vectors[:, 0], vectors[:, 1], vectors[:, 2], c='blue', s=100, alpha=0.6, label='Fruits')
ax1.scatter(mango_query[0], mango_query[1], mango_query[2], c='red', s=200, marker='*', label='Mango Query')

# Add labels for each fruit
for i, name in enumerate(names):
    ax1.text(vectors[i, 0], vectors[i, 1], vectors[i, 2], name, fontsize=8)

ax1.set_xlabel('Red')
ax1.set_ylabel('Yellow')
ax1.set_zlabel('Green')
ax1.set_title('Fruit Vectors in 3D Color Space')
ax1.legend()

# Plot 2: 2D projection (Red vs Yellow - main color dimensions)
ax2 = fig.add_subplot(122)
ax2.scatter(vectors[:, 0], vectors[:, 1], c='blue', s=100, alpha=0.6, label='Fruits')
ax2.scatter(mango_query[0], mango_query[1], c='red', s=200, marker='*', label='Mango Query')

for i, name in enumerate(names):
    ax2.annotate(name, (vectors[i, 0], vectors[i, 1]), fontsize=8)

ax2.set_xlabel('Red Intensity')
ax2.set_ylabel('Yellow Intensity')
ax2.set_title('Red vs Yellow (Top View)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Notice how fruits with similar colors cluster together!")
print("   Papaya, nectarine, peach, apricot are all in the high-red corner (top-right)")

## 3) Understanding Similarity Metrics

When searching for similar vectors, there are several ways to measure "closeness":

| Metric | How it works | Best for |
|--------|--------------|----------|
| **Cosine similarity** | Angle between vectors (ignores magnitude) | Document similarity, same direction = high similarity |
| **Euclidean distance** | Straight-line distance between points | Physical distance, when magnitude matters |
| **Dot product** | Product of magnitudes × cosine of angle | Recommendations, when larger values = more important |

Let's compute all three and compare the rankings!

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Measures the angle between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Measures the straight-line distance between two points."""
    return float(np.linalg.norm(a - b))

def dot_product(a: np.ndarray, b: np.ndarray) -> float:
    """Measures the projection of one vector onto another."""
    return float(np.dot(a, b))

# Compute all metrics
metrics_results = {
    "Cosine Similarity (higher = more similar)": sorted(
        [(name, cosine_similarity(vec, mango_query)) for name, vec in fruit_vectors.items()],
        key=lambda x: x[1], reverse=True
    ),
    "Euclidean Distance (lower = more similar)": sorted(
        [(name, euclidean_distance(vec, mango_query)) for name, vec in fruit_vectors.items()],
        key=lambda x: x[1]
    ),
    "Dot Product (higher = more similar)": sorted(
        [(name, dot_product(vec, mango_query)) for name, vec in fruit_vectors.items()],
        key=lambda x: x[1], reverse=True
    )
}

# Print detailed results
print("=" * 60)
print("📊 SIMILARITY RANKINGS COMPARISON")
print("=" * 60)

for metric_name, ranking in metrics_results.items():
    print(f"\n🔹 {metric_name}:")
    print("-" * 40)
    for rank, (name, value) in enumerate(ranking, start=1):
        print(f"  {rank}. {name:12} → {value:.4f}")

## 4) Visual Comparison of Rankings

Let's create a bar chart showing how each metric ranks the top fruits.

In [ ]:
# Get top 5 for each metric
top_cosine = [x[0] for x in metrics_results["Cosine Similarity (higher = more similar)"][:5]]
top_euclidean = [x[0] for x in metrics_results["Euclidean Distance (lower = more similar)"][:5]]
top_dot = [x[0] for x in metrics_results["Dot Product (higher = more similar)"][:5]]

print("📈 Top 5 Fruits by Each Metric:")
print("-" * 50)
print(f"{'Cosine':<20} {'Euclidean':<20} {'Dot Product':<20}")
print("-" * 50)
for i in range(5):
    print(f"{top_cosine[i]:<20} {top_euclidean[i]:<20} {top_dot[i]:<20}")

# Create a comparison visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (metric_name, ranking) in enumerate(metrics_results.items()):
    ax = axes[idx]
    top5 = ranking[:5]
    names_list = [x[0] for x in top5]
    values_list = [x[1] for x in top5]
    
    colors = ['green' if n == 'papaya' else 'steelblue' for n in names_list]
    bars = ax.barh(names_list, values_list, color=colors)
    ax.set_xlabel('Similarity Score')
    ax.set_title(metric_name.split('(')[0].strip())
    ax.invert_yaxis()
    
plt.tight_layout()
plt.show()

print("\n💡 Green bars show the TOP match - notice it's the same (papaya!) across all metrics!")

## 5) Key Insight

Notice that **papaya** is the top result across all three metrics! This makes intuitive sense - papaya and mango both have:
- High red value (0.88 vs 0.9)
- Similar yellow value (0.35 vs 0.34)
- Very low green value (0.05 vs 0.08)

While the exact ordering may change between metrics, the most similar fruit (papaya) stays consistent. This demonstrates the **robustness of vector similarity search** - even when using different mathematical approaches, semantically similar items cluster together.

In [ ]:
print("=" * 60)
print("✅ Stage 2 COMPLETE!")
print("=" * 60)
print("\n📚 What you learned:")
print("  1. Visualized fruit vectors in 3D color space")
print("  2. Compared 3 different similarity metrics")
print("  3. Saw how rankings differ but top result stays the same")
print("\n🚀 Next steps:")
print("  - Try changing the mango_query vector and see results change")
print("  - In real apps, you'd use embedding models (like sentence-transformers)")
print("    to create vectors from text, but the search concept is the same!")